# 03 · 单模态 baseline

依次训练交易、地理、对话三种模型。代码直接写出训练、预测和指标计算。

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'README.md').exists()), Path.cwd())
TARGET_COLUMNS = [f'target_{i}' for i in range(1, 5)]
PREDICTIONS = ROOT / 'artifacts' / 'predictions'; PREDICTIONS.mkdir(parents=True, exist_ok=True)
METRICS = ROOT / 'artifacts' / 'metrics'; METRICS.mkdir(parents=True, exist_ok=True)
df = pd.read_parquet(ROOT / 'data' / 'processed' / 'features.parquet')

In [26]:
ROOT / 'data' / 'processed' / 'features.parquet'

WindowsPath('c:/Users/jared/OneDrive/文档/ChatGPT/序列预测/data/processed/features.parquet')

In [16]:
df.shape, df.columns.tolist(), df[TARGET_COLUMNS].head(), len(set(df.client_id))

((12000, 46),
 ['client_id',
  'report_month',
  'target_1',
  'target_2',
  'target_3',
  'target_4',
  'fold',
  'trx__event_count',
  'trx__recency_days',
  'trx__recent_count',
  'trx__amount_sum',
  'trx__amount_mean',
  'trx__amount_std',
  'trx__amount_min',
  'trx__amount_max',
  'trx__event_type_nunique',
  'trx__event_subtype_nunique',
  'trx__currency_nunique',
  'geo__event_count',
  'geo__recency_days',
  'geo__recent_count',
  'geo__geohash_4_nunique',
  'geo__geohash_5_nunique',
  'geo__geohash_6_nunique',
  'dialog__event_count',
  'dialog__recency_days',
  'dialog__recent_count',
  'dialog__emb_mean_00',
  'dialog__emb_mean_01',
  'dialog__emb_mean_02',
  'dialog__emb_mean_03',
  'dialog__emb_mean_04',
  'dialog__emb_mean_05',
  'dialog__emb_mean_06',
  'dialog__emb_mean_07',
  'dialog__emb_mean_08',
  'dialog__emb_mean_09',
  'dialog__emb_mean_10',
  'dialog__emb_mean_11',
  'dialog__emb_mean_12',
  'dialog__emb_mean_13',
  'dialog__emb_mean_14',
  'dialog__emb_mean_1

In [17]:
train = df[df['fold'] != 4].copy()
test = df[df['fold'] == 4].copy()
print('train shape:', train.shape, 'test shape:', test.shape)


train shape: (9600, 46) test shape: (2400, 46)


In [21]:
from sklearn.linear_model import LogisticRegression
from sklearn.multioutput import MultiOutputClassifier
info_modals = ["trx", "geo", "dialog"]
results = []

In [ ]:
## apply processing pipeline to different modals since they have different missing value patterns
num_pipe = Pipeline([("imputer", SimpleImputer(strategy="median", keep_empty_features=True)),
            ("scaler", StandardScaler())])

present_pipe = SimpleImputer(strategy="constant", fill_value=0, keep_empty_features=True) ## don't fill the presence columns
for modals in info_modals:
    mod_cols = [c for c in df.columns if c.startswith(modals + "__")]
    present_cols = [c for c in mod_cols if c.endswith("__present")]
    num_cols = [c for c in mod_cols if c not in present_cols]
    
    preprocessor = ColumnTransformer([
        ("num", num_pipe, num_cols),
        ("present", present_pipe, present_cols)
    ], remainder="drop")
    
    X_train = preprocessor.fit_transform(train)
    X_test = preprocessor.transform(test)
    
    y_train = train[TARGET_COLUMNS].values
    y_test = test[TARGET_COLUMNS].values
    
    model = MultiOutputClassifier(LogisticRegression(max_iter=1000)) ## 4 class MULTI-OUTPUT classification
    model.fit(X_train, y_train)
    
    probs = np.column_stack([p[:, 1] for p in model.predict_proba(X_test)])
    
    auc_score = roc_auc_score(y_test, probs)
    ap_score = average_precision_score(y_test, probs)

    aucs, aps = [], []
    for j in range(probs.shape[1]):
        y_col = y_test[:, j]
        p_col = probs[:, j]
        if np.unique(y_col).size > 1:
            aucs.append(roc_auc_score(y_col, p_col))
        if y_col.sum() > 0:
            aps.append(average_precision_score(y_col, p_col))
    auc_score = float(np.mean(aucs))
    ap_score = float(np.mean(aps))

    
    results.append({
        "modality": modals,
        "roc_auc": auc_score,
        "average_precision": ap_score
    })

summary_df = pd.DataFrame(results).sort_values("roc_auc", ascending=False).reset_index(drop=True)
summary_df

,modality,roc_auc,average_precision
0,dialog,0.532944,0.044680
1,trx,0.518866,0.045289
2,geo,0.498305,0.044310
